# 📘 Slowly Changing Dimensions (SCD) Masterclass
## Databricks Notebook — Complete SCD Implementation Guide

**What you'll learn:**
- SCD Types 0, 1, 2, 3, and 6 (Hybrid)
- Multiple implementation patterns with MERGE
- Surrogate keys, point-in-time queries
- Production patterns: batch processing, late-arriving data, soft deletes
- NEW Insurance & HR dataset for realistic SCD scenarios

**Prerequisites:** Notebooks 01-04

---

---
# 🏗️ Insurance & HR Dataset Creation

In [ ]:
spark.sql("USE techmart_dw")
print(f"✅ techmart_dw: {spark.sql('SELECT COUNT(*) FROM customers').collect()[0][0]:,} customers")

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS insurance_hr_dw;
USE insurance_hr_dw;

In [ ]:
%sql
-- Policyholders (3000 rows)
DROP TABLE IF EXISTS policyholders;
CREATE TABLE policyholders (
  policyholder_id INT, name STRING, dob DATE, address STRING, city STRING,
  state STRING, email STRING, phone STRING, risk_category STRING, created_at TIMESTAMP
);
INSERT INTO policyholders
SELECT id AS policyholder_id,
  CONCAT(CASE abs(hash(id*3))%10 WHEN 0 THEN 'James' WHEN 1 THEN 'Mary' WHEN 2 THEN 'Robert' WHEN 3 THEN 'Patricia' WHEN 4 THEN 'John' WHEN 5 THEN 'Jennifer' WHEN 6 THEN 'Michael' WHEN 7 THEN 'Linda' WHEN 8 THEN 'David' ELSE 'Elizabeth' END, ' ', CASE abs(hash(id*7))%8 WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams' WHEN 3 THEN 'Brown' WHEN 4 THEN 'Jones' WHEN 5 THEN 'Garcia' WHEN 6 THEN 'Miller' ELSE 'Davis' END) AS name,
  date_add('1950-01-01', abs(hash(id*11))%18000) AS dob,
  CONCAT(CAST(abs(hash(id*13))%9999+1 AS STRING), ' ', CASE abs(hash(id*17))%5 WHEN 0 THEN 'Oak St' WHEN 1 THEN 'Maple Ave' WHEN 2 THEN 'Pine Rd' WHEN 3 THEN 'Elm Dr' ELSE 'Cedar Ln' END) AS address,
  CASE abs(hash(id*19))%8 WHEN 0 THEN 'New York' WHEN 1 THEN 'Chicago' WHEN 2 THEN 'Houston' WHEN 3 THEN 'Phoenix' WHEN 4 THEN 'Dallas' WHEN 5 THEN 'Austin' WHEN 6 THEN 'Denver' ELSE 'Seattle' END AS city,
  CASE abs(hash(id*23))%8 WHEN 0 THEN 'NY' WHEN 1 THEN 'IL' WHEN 2 THEN 'TX' WHEN 3 THEN 'AZ' WHEN 4 THEN 'TX' WHEN 5 THEN 'TX' WHEN 6 THEN 'CO' ELSE 'WA' END AS state,
  CONCAT('policy', id, '@insurance.com') AS email,
  CONCAT('+1-555-', LPAD(CAST(abs(hash(id*29))%10000 AS STRING),4,'0')) AS phone,
  CASE abs(hash(id*31))%4 WHEN 0 THEN 'low' WHEN 1 THEN 'medium' WHEN 2 THEN 'high' ELSE 'very_high' END AS risk_category,
  CAST(date_add('2015-01-01', abs(hash(id*37))%3000) AS TIMESTAMP) AS created_at
FROM (SELECT explode(sequence(1, 3000)) AS id);

In [ ]:
%sql
-- Policies (8000 rows)
DROP TABLE IF EXISTS policies;
CREATE TABLE policies (
  policy_id INT, policyholder_id INT, policy_type STRING, premium DECIMAL(10,2),
  coverage_amount DECIMAL(12,2), start_date DATE, end_date DATE, status STRING, agent_id INT
);
INSERT INTO policies
SELECT id AS policy_id,
  abs(hash(id*3))%3000+1 AS policyholder_id,
  CASE abs(hash(id*7))%4 WHEN 0 THEN 'auto' WHEN 1 THEN 'home' WHEN 2 THEN 'life' ELSE 'health' END AS policy_type,
  CAST(abs(hash(id*11))%5000+200 AS DECIMAL(10,2)) AS premium,
  CAST(abs(hash(id*13))%500000+10000 AS DECIMAL(12,2)) AS coverage_amount,
  date_add('2020-01-01', abs(hash(id*17))%1200) AS start_date,
  date_add('2020-01-01', abs(hash(id*17))%1200+365) AS end_date,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'active' WHEN 1 THEN 'active' WHEN 2 THEN 'lapsed' ELSE 'cancelled' END AS status,
  abs(hash(id*23))%50+1 AS agent_id
FROM (SELECT explode(sequence(1, 8000)) AS id);

In [ ]:
%sql
-- Coverage Changes (15000 rows) - simulates attribute changes over time
DROP TABLE IF EXISTS coverage_changes;
CREATE TABLE coverage_changes (
  change_id INT, policy_id INT, change_date DATE, change_type STRING,
  old_value STRING, new_value STRING, reason STRING, effective_date DATE, processed_by STRING
);
INSERT INTO coverage_changes
SELECT id AS change_id,
  abs(hash(id*3))%8000+1 AS policy_id,
  date_add('2021-01-01', abs(hash(id*7))%900) AS change_date,
  CASE abs(hash(id*11))%5 WHEN 0 THEN 'premium_change' WHEN 1 THEN 'coverage_change' WHEN 2 THEN 'address_change' WHEN 3 THEN 'beneficiary_change' ELSE 'status_change' END AS change_type,
  CONCAT('old_val_', abs(hash(id*13))%100) AS old_value,
  CONCAT('new_val_', abs(hash(id*17))%100) AS new_value,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'annual_review' WHEN 1 THEN 'customer_request' WHEN 2 THEN 'risk_reassessment' ELSE 'regulatory' END AS reason,
  date_add('2021-01-01', abs(hash(id*7))%900+abs(hash(id*23))%30) AS effective_date,
  CONCAT('agent_', abs(hash(id*29))%50) AS processed_by
FROM (SELECT explode(sequence(1, 15000)) AS id);

In [ ]:
%sql
-- Employees HR (500 rows)
DROP TABLE IF EXISTS employees_hr;
CREATE TABLE employees_hr (
  employee_id INT, name STRING, department STRING, title STRING,
  salary DECIMAL(10,2), manager_id INT, hire_date DATE, location STRING, grade_level INT
);
INSERT INTO employees_hr
SELECT id AS employee_id,
  CONCAT('Employee_', id) AS name,
  CASE abs(hash(id*3))%6 WHEN 0 THEN 'Engineering' WHEN 1 THEN 'Sales' WHEN 2 THEN 'Marketing' WHEN 3 THEN 'Finance' WHEN 4 THEN 'Operations' ELSE 'HR' END AS department,
  CASE abs(hash(id*7))%5 WHEN 0 THEN 'Senior Engineer' WHEN 1 THEN 'Manager' WHEN 2 THEN 'Analyst' WHEN 3 THEN 'Director' ELSE 'Associate' END AS title,
  CAST(abs(hash(id*11))%150000+40000 AS DECIMAL(10,2)) AS salary,
  CASE WHEN id<=5 THEN NULL ELSE abs(hash(id*13))%5+1 END AS manager_id,
  date_add('2015-01-01', abs(hash(id*17))%3000) AS hire_date,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'NYC' WHEN 1 THEN 'SF' WHEN 2 THEN 'Chicago' ELSE 'Remote' END AS location,
  abs(hash(id*23))%10+1 AS grade_level
FROM (SELECT explode(sequence(1, 500)) AS id);

In [ ]:
%sql
-- Salary History (5000 rows)
DROP TABLE IF EXISTS salary_history;
CREATE TABLE salary_history (
  record_id INT, employee_id INT, effective_date DATE,
  old_salary DECIMAL(10,2), new_salary DECIMAL(10,2), change_reason STRING, approved_by STRING
);
INSERT INTO salary_history
SELECT id AS record_id,
  abs(hash(id*3))%500+1 AS employee_id,
  date_add('2018-01-01', abs(hash(id*7))%2000) AS effective_date,
  CAST(abs(hash(id*11))%150000+40000 AS DECIMAL(10,2)) AS old_salary,
  CAST(abs(hash(id*11))%150000+40000+abs(hash(id*13))%20000 AS DECIMAL(10,2)) AS new_salary,
  CASE abs(hash(id*17))%4 WHEN 0 THEN 'promotion' WHEN 1 THEN 'annual_review' WHEN 2 THEN 'market_adjustment' ELSE 'role_change' END AS change_reason,
  CONCAT('mgr_', abs(hash(id*19))%20) AS approved_by
FROM (SELECT explode(sequence(1, 5000)) AS id);

In [ ]:
%sql
-- Department Transfers (2000 rows)
DROP TABLE IF EXISTS dept_transfers;
CREATE TABLE dept_transfers (
  transfer_id INT, employee_id INT, from_dept STRING, to_dept STRING,
  from_title STRING, to_title STRING, transfer_date DATE, reason STRING
);
INSERT INTO dept_transfers
SELECT id AS transfer_id,
  abs(hash(id*3))%500+1 AS employee_id,
  CASE abs(hash(id*7))%6 WHEN 0 THEN 'Engineering' WHEN 1 THEN 'Sales' WHEN 2 THEN 'Marketing' WHEN 3 THEN 'Finance' WHEN 4 THEN 'Operations' ELSE 'HR' END AS from_dept,
  CASE abs(hash(id*11))%6 WHEN 0 THEN 'Engineering' WHEN 1 THEN 'Sales' WHEN 2 THEN 'Marketing' WHEN 3 THEN 'Finance' WHEN 4 THEN 'Operations' ELSE 'HR' END AS to_dept,
  CASE abs(hash(id*13))%5 WHEN 0 THEN 'Associate' WHEN 1 THEN 'Analyst' WHEN 2 THEN 'Manager' WHEN 3 THEN 'Senior Engineer' ELSE 'Director' END AS from_title,
  CASE abs(hash(id*17))%5 WHEN 0 THEN 'Manager' WHEN 1 THEN 'Senior Engineer' WHEN 2 THEN 'Director' WHEN 3 THEN 'VP' ELSE 'Analyst' END AS to_title,
  date_add('2019-01-01', abs(hash(id*19))%1800) AS transfer_date,
  CASE abs(hash(id*23))%3 WHEN 0 THEN 'promotion' WHEN 1 THEN 'restructuring' ELSE 'lateral_move' END AS reason
FROM (SELECT explode(sequence(1, 2000)) AS id);

In [ ]:
%sql
SELECT 'policyholders' AS tbl, COUNT(*) AS rows FROM policyholders
UNION ALL SELECT 'policies', COUNT(*) FROM policies
UNION ALL SELECT 'coverage_changes', COUNT(*) FROM coverage_changes
UNION ALL SELECT 'employees_hr', COUNT(*) FROM employees_hr
UNION ALL SELECT 'salary_history', COUNT(*) FROM salary_history
UNION ALL SELECT 'dept_transfers', COUNT(*) FROM dept_transfers
ORDER BY tbl;

---
# 📗 Section 1: SCD Theory

## The Problem

> "What was the customer's address **when** they placed this order?"

If you overwrite the address, you lose history. SCDs solve this.

## SCD Types Overview

| Type | Strategy | History | Use Case |
|------|----------|---------|----------|
| **Type 0** | Never update | Original preserved | Registration date, SSN |
| **Type 1** | Overwrite | No history | Error corrections, non-critical |
| **Type 2** | New row per change | Full history | Address, status, risk level |
| **Type 3** | Previous value column | One level back | Previous salary, previous dept |
| **Type 6** | Hybrid (1+2+3) | Full + current overwrite | Complex reporting needs |

## Key Columns for SCD Type 2

```
┌─────────────────────────────────────────────────────────────────┐
│  surrogate_key  │  natural_key  │  attributes...  │  SCD cols   │
├─────────────────┼───────────────┼─────────────────┼─────────────┤
│  customer_sk    │  customer_id  │  name, address  │  eff_start  │
│  (unique per    │  (business    │  city, state    │  eff_end    │
│   version)      │   identifier) │                 │  is_current │
└─────────────────┴───────────────┴─────────────────┴─────────────┘
```

## SCD Decision Framework

### When to Use Each Type

| SCD Type | Use When | Example | Trade-off |
|----------|----------|---------|-----------|
| **Type 0** | Value should NEVER change | Birth date, SSN | Simplest, but inflexible |
| **Type 1** | Only current value matters | Fix a typo in name | Loses history |
| **Type 2** | Need FULL history | Customer address changes | Complex, table grows |
| **Type 3** | Need current + previous only | Previous + current address | Limited history (only 1 prior) |
| **Type 6** | Need current flag + full history | Hybrid reporting needs | Most complex |

### Real-World SCD Examples

| Dimension | Attribute | SCD Type | Why |
|-----------|-----------|----------|-----|
| Customer | Name | Type 1 | Typo fix, don't need history of typos |
| Customer | Address/City | Type 2 | Need to know where they lived for each order |
| Customer | Email | Type 1 | Only current email matters for communication |
| Customer | Segment (Gold/Silver) | Type 2 | Need to track when they were upgraded |
| Product | Price | Type 2 | Need historical prices for revenue accuracy |
| Product | Description | Type 1 | Only current description matters |
| Employee | Salary | Type 2 | Need history for compensation analysis |
| Employee | Department | Type 2 | Need to know which dept they were in |
| Employee | Phone number | Type 1 | Only current matters |

### The SCD Type 2 Table Structure

```sql
CREATE TABLE dim_customer (
    -- Surrogate key (auto-generated, unique per VERSION)
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    
    -- Business key (natural key from source — NOT unique in this table!)
    customer_id INT,
    
    -- Dimension attributes (the values that change)
    name STRING,
    city STRING,
    segment STRING,
    
    -- SCD Type 2 metadata columns
    effective_from DATE,        -- When this version became active
    effective_to DATE,          -- When this version was superseded (9999-12-31 = current)
    is_current BOOLEAN,         -- Quick filter for current version
    
    -- Audit columns
    _loaded_at TIMESTAMP,
    _source STRING
)
```

> 💡 **Key Insight:** In SCD Type 2, the business key (customer_id) is NOT unique!
> A customer with 3 address changes has 3 rows. The surrogate key (customer_sk) IS unique.

In [ ]:
# SCD Type comparison — same scenario, different approaches
print("📊 SCD TYPE COMPARISON — Same Change, Different Approaches")
print("=" * 70)

# Scenario: Customer Alice moves from NYC to San Francisco on 2024-03-15
print("""
SCENARIO: Alice (customer_id=1) moves from NYC to San Francisco on 2024-03-15

┌─────────────────────────────────────────────────────────────────────────┐
│ TYPE 0 (Retain Original):                                                │
│   customer_id=1, city='NYC'  ← NEVER changes, even though she moved     │
│   (Use for: birth_date, original_signup_date)                           │
├─────────────────────────────────────────────────────────────────────────┤
│ TYPE 1 (Overwrite):                                                      │
│   BEFORE: customer_id=1, city='NYC'                                      │
│   AFTER:  customer_id=1, city='San Francisco'  ← NYC is GONE forever    │
│   (Use for: email, phone — only current matters)                        │
├─────────────────────────────────────────────────────────────────────────┤
│ TYPE 2 (Add Row):                                                        │
│   sk=1, customer_id=1, city='NYC',           eff_from=2020-01-01,       │
│         eff_to=2024-03-14, is_current=FALSE                             │
│   sk=2, customer_id=1, city='San Francisco', eff_from=2024-03-15,       │
│         eff_to=9999-12-31, is_current=TRUE                              │
│   (Use for: address, segment — need history for analytics)              │
├─────────────────────────────────────────────────────────────────────────┤
│ TYPE 3 (Add Column):                                                     │
│   customer_id=1, current_city='San Francisco', previous_city='NYC'      │
│   (Use for: when you only need current + one previous value)            │
├─────────────────────────────────────────────────────────────────────────┤
│ TYPE 6 (Hybrid 1+2+3):                                                  │
│   sk=1, customer_id=1, city='San Francisco', historical_city='NYC',     │
│         eff_from=2020-01-01, eff_to=2024-03-14, is_current=FALSE        │
│   sk=2, customer_id=1, city='San Francisco', historical_city='SF',      │
│         eff_from=2024-03-15, eff_to=9999-12-31, is_current=TRUE         │
│   (Type 1 overwrites 'city' in ALL rows + Type 2 history + Type 3 prev) │
└─────────────────────────────────────────────────────────────────────────┘
""")


---
# 📗 Section 2: SCD Type 0 (Retain Original)

**Rule:** Never update certain columns. Keep the original value forever.

**Use cases:** Registration date, first purchase date, original credit score

In [ ]:
%sql
-- SCD Type 0: Protect original registration date
-- Even if source sends a "corrected" date, we keep the original
USE insurance_hr_dw;

DROP TABLE IF EXISTS dim_policyholder_type0;
CREATE TABLE dim_policyholder_type0 AS
SELECT policyholder_id, name, address, city, state, risk_category,
  created_at AS original_registration_date,
  current_timestamp() AS _loaded_at
FROM policyholders;

SELECT * FROM dim_policyholder_type0 LIMIT 5;

In [ ]:
%sql
-- Simulate an update that tries to change registration date
-- Type 0: MERGE that protects the original_registration_date
MERGE INTO dim_policyholder_type0 AS target
USING (
  SELECT policyholder_id, name, 'NEW ADDRESS 123' AS address, city, state,
    'high' AS risk_category, '2024-01-01' AS created_at
  FROM policyholders WHERE policyholder_id <= 5
) AS source
ON target.policyholder_id = source.policyholder_id
WHEN MATCHED THEN UPDATE SET
  target.name = source.name,
  target.address = source.address,
  target.risk_category = source.risk_category
  -- NOTE: original_registration_date is NOT updated (Type 0)
WHEN NOT MATCHED THEN INSERT *;

-- Verify: registration date unchanged, address updated
SELECT policyholder_id, address, original_registration_date FROM dim_policyholder_type0 WHERE policyholder_id <= 5;

---
# 📗 Section 3: SCD Type 1 (Overwrite)

**Rule:** Simply replace old value with new. No history kept.

**Use cases:** Fixing typos, updating email, non-critical attributes

In [ ]:
%sql
-- SCD Type 1: Overwrite employee title after promotion
DROP TABLE IF EXISTS dim_employee_type1;
CREATE TABLE dim_employee_type1 AS
SELECT employee_id, name, department, title, salary, location, grade_level,
  current_timestamp() AS _last_updated
FROM employees_hr;

-- Before
SELECT employee_id, name, title, salary FROM dim_employee_type1 WHERE employee_id <= 5;

In [ ]:
%sql
-- Apply promotions (Type 1 overwrite)
MERGE INTO dim_employee_type1 AS target
USING (
  SELECT employee_id, to_title AS new_title, to_dept AS new_dept
  FROM dept_transfers
  WHERE employee_id <= 5
  ORDER BY transfer_date DESC
  LIMIT 5
) AS source
ON target.employee_id = source.employee_id
WHEN MATCHED THEN UPDATE SET
  target.title = source.new_title,
  target.department = source.new_dept,
  target._last_updated = current_timestamp();

-- After: title overwritten, no history of old title
SELECT employee_id, name, title, department, _last_updated FROM dim_employee_type1 WHERE employee_id <= 5;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: SCD Type 1
-- ============================================
-- Update policyholder email addresses (Type 1):
-- 1. Create dim_policyholder_type1 from policyholders
-- 2. MERGE with new emails for policyholder_id 1-10
--    (new email = CONCAT('updated_', policyholder_id, '@new.com'))
-- 3. Verify the overwrite worked
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
DROP TABLE IF EXISTS dim_policyholder_type1;
CREATE TABLE dim_policyholder_type1 AS
SELECT * FROM policyholders;

MERGE INTO dim_policyholder_type1 AS target
USING (
  SELECT id AS policyholder_id, CONCAT('updated_', id, '@new.com') AS new_email
  FROM (SELECT explode(sequence(1, 10)) AS id)
) AS source
ON target.policyholder_id = source.policyholder_id
WHEN MATCHED THEN UPDATE SET target.email = source.new_email;

SELECT policyholder_id, email FROM dim_policyholder_type1 WHERE policyholder_id <= 10;

---
# 📗 Section 4: SCD Type 2 (Full History) — THE MAIN EVENT

**Rule:** Create a new row for every change. Old row gets an end date.

**Key columns:**
- `surrogate_key` — unique per version (not per entity)
- `effective_start_date` — when this version became active
- `effective_end_date` — when this version was superseded (9999-12-31 = current)
- `is_current` — boolean flag for easy filtering

This is the most important SCD type for Data Engineering interviews.

In [ ]:
%sql
-- SCD Type 2: Initial load of policyholder dimension
DROP TABLE IF EXISTS dim_policyholder_scd2;
CREATE TABLE dim_policyholder_scd2 (
  policyholder_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  policyholder_id INT,
  name STRING,
  address STRING,
  city STRING,
  state STRING,
  risk_category STRING,
  effective_start_date DATE,
  effective_end_date DATE,
  is_current BOOLEAN,
  _row_hash STRING
);

-- Initial load: all records are current
INSERT INTO dim_policyholder_scd2 (policyholder_id, name, address, city, state, risk_category, effective_start_date, effective_end_date, is_current, _row_hash)
SELECT policyholder_id, name, address, city, state, risk_category,
  CAST(created_at AS DATE) AS effective_start_date,
  CAST('9999-12-31' AS DATE) AS effective_end_date,
  true AS is_current,
  MD5(CONCAT(name, address, city, state, risk_category)) AS _row_hash
FROM policyholders;

SELECT COUNT(*) AS initial_load FROM dim_policyholder_scd2;

In [ ]:
%sql
-- Simulate incoming changes (address updates for 100 policyholders)
DROP TABLE IF EXISTS policyholder_changes;
CREATE TABLE policyholder_changes AS
SELECT policyholder_id, name,
  CONCAT(CAST(abs(hash(policyholder_id * 99))%9999 AS STRING), ' New Blvd') AS address,
  'Los Angeles' AS city, 'CA' AS state, risk_category,
  CAST('2024-06-01' AS DATE) AS change_date
FROM policyholders
WHERE policyholder_id <= 100;

SELECT COUNT(*) AS changes_to_apply FROM policyholder_changes;

In [ ]:
%sql
-- SCD Type 2: Apply changes (Step 1: Close old records)
-- Close current records that have changes
MERGE INTO dim_policyholder_scd2 AS target
USING (
  SELECT pc.policyholder_id, pc.change_date,
    MD5(CONCAT(pc.name, pc.address, pc.city, pc.state, pc.risk_category)) AS new_hash
  FROM policyholder_changes pc
) AS source
ON target.policyholder_id = source.policyholder_id
  AND target.is_current = true
  AND target._row_hash != source.new_hash
WHEN MATCHED THEN UPDATE SET
  target.effective_end_date = date_sub(source.change_date, 1),
  target.is_current = false;

In [ ]:
%sql
-- SCD Type 2: Apply changes (Step 2: Insert new versions)
INSERT INTO dim_policyholder_scd2 (policyholder_id, name, address, city, state, risk_category, effective_start_date, effective_end_date, is_current, _row_hash)
SELECT policyholder_id, name, address, city, state, risk_category,
  change_date AS effective_start_date,
  CAST('9999-12-31' AS DATE) AS effective_end_date,
  true AS is_current,
  MD5(CONCAT(name, address, city, state, risk_category)) AS _row_hash
FROM policyholder_changes;

-- Verify: should have original + new versions for changed records
SELECT COUNT(*) AS total_rows, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM dim_policyholder_scd2;

In [ ]:
%sql
-- Point-in-time query: "What was policyholder 1's address on 2024-01-01?"
SELECT policyholder_id, address, city, state, effective_start_date, effective_end_date, is_current
FROM dim_policyholder_scd2
WHERE policyholder_id = 1
  AND effective_start_date <= '2024-01-01'
  AND effective_end_date >= '2024-01-01';

In [ ]:
%sql
-- View full history for a policyholder
SELECT policyholder_id, address, city, state, effective_start_date, effective_end_date, is_current
FROM dim_policyholder_scd2
WHERE policyholder_id = 1
ORDER BY effective_start_date;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: SCD Type 2 for Employees
-- ============================================
-- Build an SCD2 dimension for employees tracking department changes:
-- 1. Create dim_employee_scd2 with surrogate key, effective dates, is_current
-- 2. Initial load from employees_hr
-- 3. Apply dept_transfers as changes (use transfer_date as effective_start)
-- 4. Query: "What department was employee 1 in on 2022-01-01?"
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
DROP TABLE IF EXISTS dim_employee_scd2;
CREATE TABLE dim_employee_scd2 (
  employee_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  employee_id INT, name STRING, department STRING, title STRING,
  salary DECIMAL(10,2), location STRING,
  effective_start_date DATE, effective_end_date DATE, is_current BOOLEAN
);

-- Initial load
INSERT INTO dim_employee_scd2 (employee_id, name, department, title, salary, location, effective_start_date, effective_end_date, is_current)
SELECT employee_id, name, department, title, salary, location,
  hire_date, CAST('9999-12-31' AS DATE), true
FROM employees_hr;

-- Close records that have transfers
MERGE INTO dim_employee_scd2 AS target
USING (
  SELECT employee_id, MIN(transfer_date) AS first_transfer
  FROM dept_transfers GROUP BY employee_id
) AS source
ON target.employee_id = source.employee_id AND target.is_current = true
WHEN MATCHED THEN UPDATE SET
  target.effective_end_date = date_sub(source.first_transfer, 1),
  target.is_current = false;

-- Insert new versions from transfers
INSERT INTO dim_employee_scd2 (employee_id, name, department, title, salary, location, effective_start_date, effective_end_date, is_current)
SELECT dt.employee_id, e.name, dt.to_dept, dt.to_title, e.salary, e.location,
  dt.transfer_date, CAST('9999-12-31' AS DATE), true
FROM dept_transfers dt
JOIN employees_hr e ON dt.employee_id = e.employee_id
WHERE dt.transfer_date = (SELECT MAX(transfer_date) FROM dept_transfers d2 WHERE d2.employee_id = dt.employee_id);

-- Point-in-time query
SELECT * FROM dim_employee_scd2
WHERE employee_id = 1 AND effective_start_date <= '2022-01-01' AND effective_end_date >= '2022-01-01';

## SCD Type 2 — MERGE Edge Cases

### Edge Case 1: Multiple Changes in Same Batch

What if a customer changes city TWICE in the same batch?
```
Batch contains:
  customer_id=1, city='Chicago', timestamp=10:00
  customer_id=1, city='Denver', timestamp=11:00
```

**Solution:** Deduplicate first (keep latest), OR process in sequence order.

### Edge Case 2: Late-Arriving Changes

What if we receive a change from LAST WEEK today?
```
Current record: customer_id=1, city='Denver', eff_from=2024-03-10
Late arrival:   customer_id=1, city='Chicago', change_date=2024-03-05
```

**Solution:** Insert the late record with correct effective dates, split existing record.

### Edge Case 3: Revert to Previous Value

What if a customer moves BACK to their original city?
```
Version 1: city='NYC' (2020-2023)
Version 2: city='SF' (2023-2024)
Version 3: city='NYC' (2024-present)  ← Same as version 1!
```

**Solution:** Still create a new version. The history shows they left and came back.

### Edge Case 4: NULL Values

What if the new value is NULL? Is that a "change"?

**Solution:** Define business rules. Usually: NULL → ignore (don't create new version for missing data).

In [ ]:
# SCD Type 2 MERGE with edge case handling
print("🔀 SCD TYPE 2 — Production MERGE Pattern")
print("=" * 60)

# The complete SCD Type 2 MERGE statement
scd2_merge = """
-- PRODUCTION SCD TYPE 2 MERGE
-- Handles: new records, changed records, unchanged records (skip)

MERGE INTO dim_customer AS target
USING (
    -- Source: deduplicated, latest per business key
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY change_timestamp DESC) AS rn
        FROM staging_customer_changes
    ) WHERE rn = 1
) AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE

-- CASE 1: Existing customer with CHANGED attributes → close old, insert new
WHEN MATCHED 
    AND (target.city != source.city OR target.segment != source.segment)
    THEN UPDATE SET
        target.effective_to = source.change_date,
        target.is_current = FALSE

-- CASE 2: New customer (not in dimension) → insert
WHEN NOT MATCHED THEN INSERT (
    customer_id, name, city, segment, effective_from, effective_to, is_current
) VALUES (
    source.customer_id, source.name, source.city, source.segment,
    source.change_date, '9999-12-31', TRUE
)
;

-- STEP 2: Insert the NEW version for changed records
-- (MERGE can only do one action per matched row, so we need a second statement)
INSERT INTO dim_customer (customer_id, name, city, segment, effective_from, effective_to, is_current)
SELECT 
    source.customer_id, source.name, source.city, source.segment,
    source.change_date, '9999-12-31', TRUE
FROM staging_customer_changes source
JOIN dim_customer target 
    ON source.customer_id = target.customer_id
    AND target.effective_to = source.change_date  -- Just closed by the MERGE above
    AND target.is_current = FALSE;
"""

print(scd2_merge)
print("\n💡 Key Points:")
print("   1. MERGE closes the old record (set effective_to, is_current=FALSE)")
print("   2. Separate INSERT adds the new version (is_current=TRUE)")
print("   3. Unchanged records are SKIPPED (no unnecessary updates)")
print("   4. Source is deduplicated BEFORE merge (handle multiple changes)")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Implement SCD Type 2
# ============================================
# Given this scenario:
# - dim_customer table exists with 5 customers (all is_current=TRUE)
# - A new batch arrives with 3 changes:
#   * Customer 1: city changed from 'NYC' to 'Boston'
#   * Customer 3: segment changed from 'bronze' to 'silver'
#   * Customer 6: NEW customer (not in dimension yet)
#
# Write the SQL to:
# 1. Close the old records for customers 1 and 3
# 2. Insert new versions for customers 1 and 3
# 3. Insert customer 6 as a new record
#
# Use the techmart_dw database and create test tables.


In [ ]:
%sql
-- ✅ SOLUTION: SCD Type 2 Implementation
-- Step 0: Create the dimension table
DROP TABLE IF EXISTS techmart_dw.dim_customer_scd2;
CREATE TABLE techmart_dw.dim_customer_scd2 (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    name STRING,
    city STRING,
    segment STRING,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
) USING DELTA

In [ ]:
%sql
-- Load initial dimension data
INSERT INTO techmart_dw.dim_customer_scd2 (customer_id, name, city, segment, effective_from, effective_to, is_current)
VALUES
(1, 'Alice', 'NYC', 'gold', '2020-01-01', '9999-12-31', TRUE),
(2, 'Bob', 'LA', 'silver', '2020-01-01', '9999-12-31', TRUE),
(3, 'Charlie', 'Chicago', 'bronze', '2021-06-15', '9999-12-31', TRUE),
(4, 'Diana', 'Seattle', 'gold', '2022-03-01', '9999-12-31', TRUE),
(5, 'Eve', 'Austin', 'silver', '2023-01-01', '9999-12-31', TRUE)

In [ ]:
%sql
-- Create staging table with incoming changes
DROP TABLE IF EXISTS techmart_dw.stg_customer_changes;
CREATE TABLE techmart_dw.stg_customer_changes AS
SELECT * FROM VALUES
(1, 'Alice', 'Boston', 'gold', DATE'2024-03-15'),
(3, 'Charlie', 'Chicago', 'silver', DATE'2024-03-15'),
(6, 'Frank', 'Denver', 'bronze', DATE'2024-03-15')
AS t(customer_id, name, city, segment, change_date)

In [ ]:
%sql
-- Step 1: Close old records for CHANGED customers
MERGE INTO techmart_dw.dim_customer_scd2 AS target
USING techmart_dw.stg_customer_changes AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND (target.city != source.city OR target.segment != source.segment)
THEN UPDATE SET
    effective_to = source.change_date,
    is_current = FALSE

In [ ]:
%sql
-- Step 2: Insert NEW versions for changed customers + new customers
INSERT INTO techmart_dw.dim_customer_scd2 (customer_id, name, city, segment, effective_from, effective_to, is_current)
SELECT customer_id, name, city, segment, change_date, DATE'9999-12-31', TRUE
FROM techmart_dw.stg_customer_changes

In [ ]:
%sql
-- Verify: Alice should have 2 rows, Charlie 2 rows, Frank 1 row
SELECT customer_id, name, city, segment, effective_from, effective_to, is_current
FROM techmart_dw.dim_customer_scd2
WHERE customer_id IN (1, 3, 6)
ORDER BY customer_id, effective_from

---
# 📗 Section 5: SCD Type 3 (Previous Value Column)

**Rule:** Keep current value + one previous value in the same row.

**Use case:** When you only need one level of history (e.g., previous salary).

In [ ]:
%sql
-- SCD Type 3: Track current and previous salary
DROP TABLE IF EXISTS dim_employee_type3;
CREATE TABLE dim_employee_type3 AS
SELECT employee_id, name, department, title,
  salary AS current_salary,
  CAST(NULL AS DECIMAL(10,2)) AS previous_salary,
  CAST(NULL AS DATE) AS salary_change_date,
  current_timestamp() AS _last_updated
FROM employees_hr;

-- Apply salary changes (keep previous)
MERGE INTO dim_employee_type3 AS target
USING (
  SELECT employee_id, new_salary, effective_date
  FROM salary_history
  WHERE record_id IN (
    SELECT MAX(record_id) FROM salary_history GROUP BY employee_id
  )
) AS source
ON target.employee_id = source.employee_id
WHEN MATCHED THEN UPDATE SET
  target.previous_salary = target.current_salary,
  target.current_salary = source.new_salary,
  target.salary_change_date = source.effective_date,
  target._last_updated = current_timestamp();

SELECT employee_id, name, current_salary, previous_salary, salary_change_date
FROM dim_employee_type3 WHERE previous_salary IS NOT NULL LIMIT 10;

---
# 📗 Section 6: SCD Type 6 (Hybrid 1+2+3)

**Rule:** Combines all approaches:
- Type 2 rows (full history with effective dates)
- Type 3 column (previous value on current row)
- Type 1 overwrite (current value updated on ALL historical rows)

Most complex but most flexible for reporting.

In [ ]:
%sql
-- SCD Type 6: Hybrid for risk_category
DROP TABLE IF EXISTS dim_policyholder_type6;
CREATE TABLE dim_policyholder_type6 (
  policyholder_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  policyholder_id INT,
  name STRING,
  risk_category STRING,
  previous_risk_category STRING,
  current_risk_category STRING,
  effective_start_date DATE,
  effective_end_date DATE,
  is_current BOOLEAN
);

-- Initial load
INSERT INTO dim_policyholder_type6 (policyholder_id, name, risk_category, previous_risk_category, current_risk_category, effective_start_date, effective_end_date, is_current)
SELECT policyholder_id, name, risk_category, NULL, risk_category,
  CAST(created_at AS DATE), CAST('9999-12-31' AS DATE), true
FROM policyholders WHERE policyholder_id <= 50;

SELECT * FROM dim_policyholder_type6 WHERE policyholder_id = 1;

In [ ]:
%sql
-- Apply a risk category change (Type 6 style)
-- Step 1: Close current record (Type 2 behavior)
UPDATE dim_policyholder_type6
SET effective_end_date = '2024-05-31', is_current = false
WHERE policyholder_id = 1 AND is_current = true;

-- Step 2: Insert new version with previous value (Type 2 + Type 3)
INSERT INTO dim_policyholder_type6 (policyholder_id, name, risk_category, previous_risk_category, current_risk_category, effective_start_date, effective_end_date, is_current)
VALUES (1, 'James Smith', 'very_high', 'low', 'very_high', '2024-06-01', '9999-12-31', true);

-- Step 3: Update current_risk_category on ALL rows for this entity (Type 1 behavior)
UPDATE dim_policyholder_type6
SET current_risk_category = 'very_high'
WHERE policyholder_id = 1;

-- Result: full history + current value on every row
SELECT * FROM dim_policyholder_type6 WHERE policyholder_id = 1 ORDER BY effective_start_date;

---
# 📗 Section 7: Surrogate Keys

**WHY:** Natural keys (customer_id) identify the entity. Surrogate keys (customer_sk)
identify the *version*. Each SCD2 row gets a unique surrogate key.

In [ ]:
%sql
-- Surrogate key strategies
-- 1. IDENTITY (auto-increment) — used above
-- 2. ROW_NUMBER based
SELECT ROW_NUMBER() OVER (ORDER BY policyholder_id, effective_start_date) AS manual_sk,
  policyholder_id, address, effective_start_date, is_current
FROM dim_policyholder_scd2
WHERE policyholder_id <= 3
ORDER BY policyholder_id, effective_start_date;

---
# 📗 Section 8: Production Patterns

## 8.1 Batch SCD Processing

In [ ]:
# Batch SCD2 processor in PySpark
from pyspark.sql.functions import *
from pyspark.sql.window import Window

def apply_scd2_batch(target_table, changes_df, key_col, tracked_cols, change_date_col):
    """Apply a batch of SCD2 changes.
    
    Args:
        target_table: name of the SCD2 dimension table
        changes_df: DataFrame with incoming changes
        key_col: natural key column name
        tracked_cols: list of columns to track for changes
        change_date_col: column with the effective date of change
    """
    target = spark.table(target_table)
    
    # Build hash for change detection
    hash_expr = md5(concat_ws("|", *[col(c).cast("string") for c in tracked_cols]))
    
    changes_with_hash = changes_df.withColumn("_new_hash", hash_expr)
    current_target = target.filter("is_current = true")
    
    # Find actual changes (hash differs)
    actual_changes = (changes_with_hash.alias("src")
        .join(current_target.alias("tgt"), key_col, "inner")
        .filter("src._new_hash != tgt._row_hash")
        .select("src.*")
    )
    
    change_count = actual_changes.count()
    print(f"  Detected {change_count} actual changes out of {changes_df.count()} incoming records")
    return change_count

# Demo
spark.sql("USE insurance_hr_dw")
changes = spark.sql("SELECT policyholder_id, name, address, city, state, risk_category, change_date FROM policyholder_changes")
apply_scd2_batch("dim_policyholder_scd2", changes, "policyholder_id", 
    ["name", "address", "city", "state", "risk_category"], "change_date")

## 8.2 Late-Arriving Dimension Changes

In [ ]:
%sql
-- Late-arriving change: we discover policyholder 50 moved on 2023-03-01
-- but we only got the data now
-- This requires inserting a historical version between existing versions

-- Check current state
SELECT policyholder_id, address, effective_start_date, effective_end_date, is_current
FROM dim_policyholder_scd2
WHERE policyholder_id = 50
ORDER BY effective_start_date;

## 8.3 Soft Deletes

In [ ]:
%sql
-- Soft delete pattern: mark as deleted without removing
DROP TABLE IF EXISTS dim_policyholder_soft_delete;
CREATE TABLE dim_policyholder_soft_delete AS
SELECT *, false AS is_deleted, CAST(NULL AS TIMESTAMP) AS deleted_at
FROM policyholders WHERE policyholder_id <= 100;

-- Soft delete some records
UPDATE dim_policyholder_soft_delete
SET is_deleted = true, deleted_at = current_timestamp()
WHERE policyholder_id IN (5, 10, 15, 20);

-- Query active records only
SELECT COUNT(*) AS active_count FROM dim_policyholder_soft_delete WHERE is_deleted = false;
SELECT COUNT(*) AS deleted_count FROM dim_policyholder_soft_delete WHERE is_deleted = true;

## CDC as Input to SCD

In production, SCD changes come from **CDC (Change Data Capture)**:

```
Source DB → CDC (Debezium) → Kafka → Bronze (raw events) → SCD MERGE → Dimension
```

The CDC event tells you WHAT changed:
- `operation = 'I'` → New customer → INSERT into dimension
- `operation = 'U'` → Changed customer → Close old + INSERT new (Type 2)
- `operation = 'D'` → Deleted customer → Soft delete (set is_active=FALSE)

> 📌 See **Notebook 48 (CDC)** for full CDC pipeline implementation.
> This notebook focuses on the SCD MERGE patterns that CONSUME CDC events.

## Performance Considerations for Large Dimensions

| Dimension Size | Strategy | Why |
|---------------|----------|-----|
| < 1M rows | Simple MERGE | Fast enough, no optimization needed |
| 1M - 100M rows | Partition by is_current | Only scan current records for matching |
| > 100M rows | Partition by effective_from (monthly) | Limit scan to recent partitions |
| > 1B rows | Bucketed by business_key | Co-locate records for same customer |

### Optimizing SCD Type 2 MERGE Performance

```sql
-- Optimization 1: Only match against CURRENT records
ON target.customer_id = source.customer_id 
AND target.is_current = TRUE  -- ← This is critical! Avoids scanning history

-- Optimization 2: Liquid Clustering on business key
ALTER TABLE dim_customer CLUSTER BY (customer_id, is_current);

-- Optimization 3: Z-ORDER for legacy tables
OPTIMIZE dim_customer ZORDER BY (customer_id, is_current);
```

In [ ]:
# ============================================
# 🎯 YOUR TURN: Query SCD Type 2 History
# ============================================
# Using the dim_customer_scd2 table we created:
#
# 1. Find Alice's CURRENT city
# 2. Find Alice's city as of 2023-06-01 (point-in-time query)
# 3. Count how many times each customer's record changed
# 4. Find customers who changed segment (bronze → silver → gold progression)
#
# Write your queries below:


In [ ]:
%sql
-- ✅ SOLUTION 1: Alice's current city
SELECT customer_id, name, city, segment
FROM techmart_dw.dim_customer_scd2
WHERE customer_id = 1 AND is_current = TRUE

In [ ]:
%sql
-- ✅ SOLUTION 2: Point-in-time query (Alice's city on 2023-06-01)
SELECT customer_id, name, city, segment, effective_from, effective_to
FROM techmart_dw.dim_customer_scd2
WHERE customer_id = 1
  AND effective_from <= '2023-06-01'
  AND effective_to > '2023-06-01'

In [ ]:
%sql
-- ✅ SOLUTION 3: Count changes per customer
SELECT customer_id, name, COUNT(*) AS versions,
       COUNT(*) - 1 AS times_changed
FROM techmart_dw.dim_customer_scd2
GROUP BY customer_id, name
ORDER BY versions DESC

---
# 🚀 Section 9: Mini Projects

## Project 1: Complete Customer Dimension (SCD2)

In [ ]:
%sql
-- Full SCD2 customer dimension from techmart_dw
USE insurance_hr_dw;

DROP TABLE IF EXISTS dim_customer_complete;
CREATE TABLE dim_customer_complete (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id INT, first_name STRING, last_name STRING,
  email STRING, city STRING, state STRING, customer_segment STRING,
  lifetime_value DECIMAL(12,2),
  effective_start_date DATE, effective_end_date DATE, is_current BOOLEAN,
  _row_hash STRING, _loaded_at TIMESTAMP
);

-- Initial load
INSERT INTO dim_customer_complete (customer_id, first_name, last_name, email, city, state, customer_segment, lifetime_value, effective_start_date, effective_end_date, is_current, _row_hash, _loaded_at)
SELECT customer_id, first_name, last_name, email, city, state, customer_segment, lifetime_value,
  COALESCE(CAST(registration_date AS DATE), CAST('2020-01-01' AS DATE)),
  CAST('9999-12-31' AS DATE), true,
  MD5(CONCAT(COALESCE(first_name,''), COALESCE(city,''), COALESCE(state,''), COALESCE(customer_segment,''))),
  current_timestamp()
FROM techmart_dw.customers WHERE customer_id <= 500;

SELECT COUNT(*) AS initial_load FROM dim_customer_complete;

In [ ]:
%sql
-- Simulate Round 1 changes: 50 customers moved cities
MERGE INTO dim_customer_complete AS target
USING (
  SELECT customer_id, first_name, last_name, email, 'San Francisco' AS city, 'CA' AS state,
    customer_segment, lifetime_value, CAST('2024-03-01' AS DATE) AS change_date,
    MD5(CONCAT(COALESCE(first_name,''), 'San Francisco', 'CA', COALESCE(customer_segment,''))) AS new_hash
  FROM techmart_dw.customers WHERE customer_id BETWEEN 1 AND 50
) AS source
ON target.customer_id = source.customer_id AND target.is_current = true AND target._row_hash != source.new_hash
WHEN MATCHED THEN UPDATE SET
  target.effective_end_date = date_sub(source.change_date, 1),
  target.is_current = false;

INSERT INTO dim_customer_complete (customer_id, first_name, last_name, email, city, state, customer_segment, lifetime_value, effective_start_date, effective_end_date, is_current, _row_hash, _loaded_at)
SELECT customer_id, first_name, last_name, email, 'San Francisco', 'CA', customer_segment, lifetime_value,
  CAST('2024-03-01' AS DATE), CAST('9999-12-31' AS DATE), true,
  MD5(CONCAT(COALESCE(first_name,''), 'San Francisco', 'CA', COALESCE(customer_segment,''))),
  current_timestamp()
FROM techmart_dw.customers WHERE customer_id BETWEEN 1 AND 50;

SELECT COUNT(*) AS after_round1, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM dim_customer_complete;

## Project 2: Employee History Tracker

In [ ]:
%sql
-- Track complete employee history: dept + title + salary changes
SELECT e.employee_id, e.name,
  dt.from_dept, dt.to_dept, dt.from_title, dt.to_title, dt.transfer_date,
  sh.old_salary, sh.new_salary, sh.effective_date AS salary_date
FROM employees_hr e
LEFT JOIN dept_transfers dt ON e.employee_id = dt.employee_id
LEFT JOIN salary_history sh ON e.employee_id = sh.employee_id
  AND sh.effective_date BETWEEN date_sub(dt.transfer_date, 30) AND date_add(dt.transfer_date, 30)
WHERE e.employee_id <= 5
ORDER BY e.employee_id, dt.transfer_date;

## Project 3: Insurance Policy Dimension

In [ ]:
%sql
-- Policy dimension with coverage change history
DROP TABLE IF EXISTS dim_policy_history;
CREATE TABLE dim_policy_history AS
SELECT p.policy_id, p.policyholder_id, p.policy_type, p.premium, p.coverage_amount, p.status,
  cc.change_type, cc.old_value, cc.new_value, cc.reason,
  cc.effective_date,
  COALESCE(LEAD(cc.effective_date) OVER (PARTITION BY p.policy_id ORDER BY cc.effective_date), CAST('9999-12-31' AS DATE)) AS next_change_date
FROM policies p
LEFT JOIN coverage_changes cc ON p.policy_id = cc.policy_id
WHERE p.policy_id <= 100
ORDER BY p.policy_id, cc.effective_date;

-- Point-in-time: what was policy 1's state on 2023-06-01?
SELECT * FROM dim_policy_history
WHERE policy_id = 1 AND effective_date <= '2023-06-01' AND next_change_date > '2023-06-01';

---
# 🏆 Section 10: Interview Questions

## Q1: Explain SCD Type 2 and implement with MERGE

**Answer:** SCD Type 2 creates a new row for every change, preserving full history.
Each row has effective_start_date, effective_end_date, and is_current flag.

Implementation:
1. Compare incoming data hash with current row hash
2. If different: close current row (set end_date, is_current=false)
3. Insert new row with new values (start_date=today, end_date=9999-12-31, is_current=true)

## Q2: How do you handle late-arriving dimension changes?

**Answer:**
1. Identify the correct time slot for the late change
2. Split the existing version that spans that period
3. Insert the late-arriving version between the split
4. Reprocess any facts that joined to the wrong dimension version

## Q3: Type 1 vs 2 vs 3 — when to use each?

| Scenario | Type | Why |
|----------|------|-----|
| Fix a typo in customer name | Type 1 | Error correction, no history needed |
| Customer moves to new address | Type 2 | Need to know address at time of each order |
| Employee gets a raise | Type 3 | Only need current + previous salary |
| Regulatory audit trail | Type 2 | Must preserve complete history |
| Non-critical metadata | Type 1 | History adds no business value |

## Q4: How do you query "as of" a specific date?

```sql
SELECT * FROM dim_customer_scd2
WHERE customer_id = 123
  AND effective_start_date <= '2023-06-15'
  AND effective_end_date >= '2023-06-15'
```

## Q5: Multiple changes in one batch?

**Answer:**
1. Order changes by timestamp within each entity
2. Process sequentially: each change closes the previous
3. Only the LAST change in the batch gets is_current=true
4. Use ROW_NUMBER to identify the latest per entity

---
# ✅ Notebook Complete!

**What was covered:**
- SCD Type 0: Retain original (protected columns)
- SCD Type 1: Overwrite (no history)
- SCD Type 2: Full history (effective dates, surrogate keys)
- SCD Type 3: Previous value column
- SCD Type 6: Hybrid (1+2+3 combined)
- Surrogate key strategies
- Production patterns: batch processing, late-arriving data, soft deletes
- 3 mini projects: Customer Dimension, Employee Tracker, Policy History
- 5 interview questions with answers

**Database created:** `insurance_hr_dw` (6 tables)

**Next:** Notebook 06 — Delta Live Tables (Lakeflow Declarative Pipelines)

---
*All databases remain available for subsequent notebooks.*